# 🌡️ **Thermo-physiological Indices**
---

### **Learning Goals:**
- 🎯 **Goal 1 | Physiologically Equivalent Temperature (PET)**: Compute hourly PET values and classify diurnal thermal stress for four days spanning the four seasons of 2012 in Ioannina
- 🎯 **Goal 2 | Universal Thermal Climate Index (UTCI)**: Screen meteorological inputs against operational validity bounds, compute hourly UTCI values, and classify thermal stress for the same four days of 2012 in Ioannina
- 🎯 **Goal 3 | Comparing PET and UTCI**: Quantify numerical and stress-category differences between PET and UTCI and assess sensitivity to outdoor radiative load
- 🎯 **Goal 4 | Gridded UTCI Over Europe**: Compute gridded UTCI over Europe from ECMWF open-data forecasts, map thermal stress categories, and print summary statistics for Greece
- 🎯 **Goal 5 | Modified Physiologically Equivalent Temperature (mPET)**: Compare hourly mPET diurnal cycles across six demographic and occupational population groups and three outdoor worker activity levels for the same four days of 2012 in Ioannina
- 🎯 **Goal 6 | Thermal Bioclimate Diagrams Across Population Groups**: Compute monthly mPET thermal stress class frequencies across 2012 in Ioannina and compare annual thermal stress distributions across the six population groups

**Run the tutorial via Binder:**

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/one-weather-lab/biometeorology-air-quality-and-human-health/HEAD?labpath=notebooks%2F02_thermophysiological_indices.ipynb)

## ⚙️ Setup and Configuration
---
The two cells below import the required packages, set the local data paths and seasonal dates for Ioannina, and define the ECMWF open-data forecast and domain parameters.

In [ ]:
from importlib.metadata import version
from pathlib import Path
import sys

# Ensure the repository root is on sys.path when running from notebooks/
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils import (
    PET_SCALE,
    UTCI_SCALE,
    classify,
    compute_europe_utci,
    compute_pet,
    compute_utci,
    fetch_open_data_fields,
    format_activity_spread,
    format_comparison,
    format_europe_summary,
    format_mpet_daily_table,
    format_pet_class_counts,
    format_pet_daily_table,
    format_radiative_load,
    format_utci_class_counts,
    format_utci_screening,
    group_class_frequencies,
    plot_europe_utci_map,
    plot_mpet_bioclimate,
    plot_mpet_groups_diurnal,
    plot_mpet_worker_activity,
    plot_pet_diurnal,
    plot_pet_utci_comparison,
    plot_utci_diurnal,
    read_four_seasonal_days,
    read_hourly_mpet,
)

for pkg in ("biometeo", "numpy", "pandas", "matplotlib"):
    print(f"{pkg:12s} {version(pkg)}")

In [ ]:
DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Expected data directory at {DATA_DIR}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEASONAL_DATES = {
    "2012-01-31": "Winter (31 Jan 2012)",
    "2012-04-26": "Spring (26 Apr 2012)",
    "2012-07-11": "Summer (11 Jul 2012)",
    "2012-10-02": "Autumn (2 Oct 2012)",
}

# ECMWF open-data settings for Section 4 (used in the gridded Europe workflow)
RUN_DATE = "20260623"   # None = latest live run; "YYYYMMDD" = retrospective Azure mirror
RUN_TIME = 0            # [UTC hour] forecast cycle time
FORECAST_STEP_H = 12    # [h] valid at 12 UTC
RADIATION_WINDOW_H = 3  # [h] step differencing window (9 h to 12 h)
EUROPE_BOX = (20.0, 75.0, -30.0, 60.0)  # (south, north, west, east) [deg]
STATS_REGIONS = {
    "Greece": (34.5, 42.0, 19.0, 29.5),
    "Europe": (34.0, 72.0, -25.0, 45.0),
}
STATS_REGION = "Greece"

print(f"Data directory   : {DATA_DIR.relative_to(REPO_ROOT)}")
print(f"Output directory : {OUTPUT_DIR.relative_to(REPO_ROOT)}")
print(f"Seasonal days    : {', '.join(SEASONAL_DATES)}")
print(f"Open-data run    : {RUN_DATE or 'live'} {RUN_TIME:02d} UTC (+{FORECAST_STEP_H} h), region = {STATS_REGION}")

## Section 1: Physiologically Equivalent Temperature (PET)

In [ ]:
df_days = read_four_seasonal_days(data_dir=DATA_DIR)
df_days["pet"] = compute_pet(df=df_days)
print(format_pet_daily_table(df=df_days, season_labels=SEASONAL_DATES))

In [ ]:
df_days["pet_class"] = classify(values=df_days["pet"], scale=PET_SCALE)
print(format_pet_class_counts(df=df_days, season_labels=SEASONAL_DATES))
pet_fig_path = plot_pet_diurnal(
    df=df_days,
    scale=PET_SCALE,
    output_path=OUTPUT_DIR / "fig_1.2_pet_diurnal.png",
    season_labels=SEASONAL_DATES,
)

## Section 2: Universal Thermal Climate Index (UTCI)

In [ ]:
df_days["utci"] = compute_utci(df=df_days)
print(format_utci_screening(df=df_days, season_labels=SEASONAL_DATES))

In [ ]:
df_days["utci_class"] = classify(values=df_days["utci"], scale=UTCI_SCALE)
print(format_utci_class_counts(df=df_days, season_labels=SEASONAL_DATES))
utci_fig_path = plot_utci_diurnal(
    df=df_days,
    scale=UTCI_SCALE,
    output_path=OUTPUT_DIR / "fig_2.2_utci_diurnal.png",
    season_labels=SEASONAL_DATES,
)

## Section 3: Comparing PET and UTCI

In [ ]:
print(format_comparison(df=df_days))
comp_fig_path = plot_pet_utci_comparison(
    df=df_days,
    output_path=OUTPUT_DIR / "fig_3.1_pet_utci_comparison.png",
    season_labels=SEASONAL_DATES,
)

In [ ]:
# Radiative sensitivity compares the calculated radiative load with no radiative load (Tmrt = Ta) at the selected UTC timestamp.
TARGET_TIMESTAMP = "2012-07-11 12:00 UTC"
print(format_radiative_load(df=df_days, timestamp=TARGET_TIMESTAMP))

## Section 4: Gridded UTCI Over Europe

In [ ]:
forecast = fetch_open_data_fields(
    run_date=RUN_DATE,
    run_time=RUN_TIME,
    forecast_step_h=FORECAST_STEP_H,
    radiation_window_h=RADIATION_WINDOW_H,
)
europe_utci = compute_europe_utci(forecast=forecast, europe_box=EUROPE_BOX)
print(format_europe_summary(forecast=forecast, europe_utci=europe_utci, region=STATS_REGION, regions=STATS_REGIONS))

In [ ]:
europe_fig_path = plot_europe_utci_map(
    utci_c=europe_utci["utci_c"],
    lat2d=forecast["lat2d"],
    lon2d=forecast["lon2d"],
    scale=UTCI_SCALE,
    output_path=OUTPUT_DIR / "fig_4.2_europe_utci.png",
    valid_label=forecast["valid_time"].strftime("%Y-%m-%d %H:%M UTC"),
)

## Section 5: Modified Physiologically Equivalent Temperature (mPET)

In [ ]:
df_year = read_hourly_mpet(data_dir=DATA_DIR)
print(f"Loaded {len(df_year):,} hourly mPET records in Ioannina.")
mpet_groups_fig_path = plot_mpet_groups_diurnal(
    df=df_year,
    scale=PET_SCALE,
    output_path=OUTPUT_DIR / "fig_5.1_mpet_groups_diurnal.png",
    season_labels=SEASONAL_DATES,
)

In [ ]:
print(format_activity_spread(df=df_year, season_labels=SEASONAL_DATES))
mpet_activity_fig_path = plot_mpet_worker_activity(
    df=df_year,
    scale=PET_SCALE,
    output_path=OUTPUT_DIR / "fig_5.2_mpet_worker_activity.png",
    season_labels=SEASONAL_DATES,
)

## Section 6: Thermal Bioclimate Diagrams Across Population Groups

In [ ]:
frequencies = group_class_frequencies(df=df_year, scale=PET_SCALE)
bioclimate_fig_path = plot_mpet_bioclimate(
    frequencies=frequencies,
    scale=PET_SCALE,
    output_path=OUTPUT_DIR / "fig_6.2_mpet_bioclimate.png",
)

---

**🦉 Crafted with wisdom at One Weather Lab (OWL)**<br>
Laboratory of Meteorology and Climatology, Physics Department, University of Ioannina<br>
Christos Giannaros <<chris.giannaros@uoi.gr>>